# AMDA Augmentation Analysis

This notebook analyses the AMDA (Appliance Magnitude-aware Data Augmentation) technique from the SIDED paper.

## AMDA Formula

```
p_i = P_total_i / P_total  # Relative contribution
S_i = s * (1 - p_i)        # Scaling factor (s = base scale)
x̃_i,t = S_i * x_i,t       # Scaled signal
```

Key insight: Appliances with higher contribution get scaled down, while underrepresented appliances get amplified.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from edge_pipeline.data.augmentation import AMDATransform
from edge_pipeline.data.sided_loader import SidedLoader, APPLIANCE_ORDER

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## Visualise AMDA Effect

In [ ]:
# Create sample appliance data with imbalanced contributions
appliances_original = np.array([8000.0, -2000.0, 3000.0, 0.0, 1000.0], dtype=np.float32)
aggregate_original = np.full(60, np.sum(appliances_original), dtype=np.float32)

# Apply AMDA with different scaling factors
scales = [1.5, 2.0, 2.5, 3.0, 4.0]
results = {}

for s in scales:
    amda = AMDATransform(base_scale=s, probability=1.0)
    _, aug_app = amda(aggregate_original.copy(), appliances_original.copy())
    results[s] = aug_app

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original distribution
x = np.arange(len(APPLIANCE_ORDER))
width = 0.15

axes[0].bar(x, np.abs(appliances_original), width=0.6, label="Original", color="gray", edgecolor="black")
axes[0].set_xticks(x)
axes[0].set_xticklabels(APPLIANCE_ORDER)
axes[0].set_ylabel("Absolute Power (W)")
axes[0].set_title("Original Appliance Distribution")

# AMDA results for different scales
colors = plt.cm.viridis(np.linspace(0, 1, len(scales)))
for i, (s, color) in enumerate(zip(scales, colors)):
    offset = (i - len(scales)/2 + 0.5) * width
    axes[1].bar(x + offset, np.abs(results[s]), width=width, label=f"s={s}", color=color, edgecolor="black")

axes[1].set_xticks(x)
axes[1].set_xticklabels(APPLIANCE_ORDER)
axes[1].set_ylabel("Absolute Power (W)")
axes[1].set_title("AMDA-Augmented Distribution by Scale Factor")
axes[1].legend()

plt.tight_layout()
plt.show()

## Scaling Factor Analysis

In [ ]:
# Calculate scaling factors for different contribution levels
contributions = np.linspace(0, 1, 100)
scales = [1.5, 2.0, 2.5, 3.0, 4.0]

fig, ax = plt.subplots(figsize=(10, 6))

for s in scales:
    scaling_factors = s * (1 - contributions)
    ax.plot(contributions * 100, scaling_factors, label=f"s = {s}", linewidth=2)

ax.axhline(y=1.0, color="black", linestyle="--", alpha=0.5, label="No scaling")
ax.set_xlabel("Relative Contribution (%)")
ax.set_ylabel("Scaling Factor")
ax.set_title("AMDA Scaling Factor vs Relative Contribution")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Effect on Real Data

In [ ]:
# Load real data
loader = SidedLoader()
facilities = loader.load_all()
facility = facilities[0]

# Sample 1000 random windows
n_samples = 1000
indices = np.random.choice(len(facility.aggregate) - 60, n_samples)

original_apps = facility.appliances[indices]

amda = AMDATransform(base_scale=2.5, probability=1.0)
augmented_apps = []

for i in indices:
    agg = facility.aggregate[i:i+60]
    app = facility.appliances[i]
    _, aug_app = amda(agg, app)
    augmented_apps.append(aug_app)

augmented_apps = np.array(augmented_apps)

In [ ]:
# Compare distributions
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, name in enumerate(APPLIANCE_ORDER):
    # Original
    axes[0, i].hist(original_apps[:, i], bins=30, alpha=0.7, edgecolor="black")
    axes[0, i].set_title(f"{name} (Original)")
    if i == 0:
        axes[0, i].set_ylabel("Count")
    
    # Augmented
    axes[1, i].hist(augmented_apps[:, i], bins=30, alpha=0.7, edgecolor="black", color="orange")
    axes[1, i].set_title(f"{name} (AMDA s=2.5)")
    axes[1, i].set_xlabel("Power (W)")
    if i == 0:
        axes[1, i].set_ylabel("Count")

plt.suptitle("Distribution Comparison: Original vs AMDA-Augmented", y=1.02)
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
import pandas as pd

stats = []
for i, name in enumerate(APPLIANCE_ORDER):
    stats.append({
        "Appliance": name,
        "Original Mean": np.mean(original_apps[:, i]),
        "Original Std": np.std(original_apps[:, i]),
        "AMDA Mean": np.mean(augmented_apps[:, i]),
        "AMDA Std": np.std(augmented_apps[:, i]),
        "Mean Change": (np.mean(augmented_apps[:, i]) - np.mean(original_apps[:, i])) / (np.abs(np.mean(original_apps[:, i])) + 1e-8) * 100,
    })

df = pd.DataFrame(stats)
print(df.to_string(index=False))